In [0]:
from pyspark.sql.functions import col,initcap,lit

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run /Workspace/Users/chrknov6@hotmail.com/formula1/Incremental/00.Configurations

In [0]:
%run "/Workspace/Users/chrknov6@hotmail.com/formula1/Incremental/002.silver helper functions"

In [0]:
bronze_table = f'{catalog}.{bronze_schema}.results'
silver_table = f'{catalog}.{silver_schema}.results'

In [0]:
results_df = (
               spark.table(bronze_table)
                    .filter(col("batch_id") == lit(v_batch_id))
                    .drop(col("url"))
                    .withColumnsRenamed(
                        {"constructorId":"constructor_id",
                         "driverId":"driver_id",
                         "raceName":"race_name",
                         "positionText":"finish_position_text",
                         "date":"race_date",
                         "grid":"grid_position",
                         "laps":"completed_laps",
                         "number":"car_number",
                         "position":"finish_position"}
                    )
                    .filter(
                        col("season").isNotNull() & col("round").isNotNull() & 
                        col("constructor_id").isNotNull() & col("driver_id").isNotNull())
                    .dropDuplicates(["season","round","constructor_id","driver_id"])
                    .withColumn("race_name",initcap(col("race_name")))
)

In [0]:
write_to_silver(
    input_df= results_df,
    table_name= silver_table,
    merge_condition=((col("s.season") == col("t.season")) & (col("s.round") == col("t.round")) & (col("s.constructor_id") == col("t.constructor_id")) & (col("s.driver_id") == col("t.driver_id"))),
    columns_to_update=["race_date","grid_position","completed_laps","car_number","points","finish_position","finish_position_text","race_name","ingestion_time","filename","status"]
)